# Análise interativa - Solução 3 dados sintéticos e detector leve

**Autor:** Manoel Furtado  
**Objetivo:** estudar a geração de pseudo-rótulos e aumentações usada em `solucao_3_dados_sinteticos_detector_leve.py`.

Este notebook mostra como:

1. carregar imagens do Desafio 1;
2. gerar caixas fracas com OpenCV;
3. visualizar filtros e contornos candidatos;
4. converter caixas para formato YOLO;
5. testar aumentações simples;
6. ajustar parâmetros antes de criar o dataset final;
7. iniciar opcionalmente um treino YOLO leve.

## 1. Importações e caminhos

A célula localiza a pasta da solução, importa o script original e define os diretórios de entrada e saída.

In [ ]:
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

SCRIPT_NAME = 'solucao_3_dados_sinteticos_detector_leve.py'

def find_solution_dir(script_name):
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / script_name).exists():
            return candidate
    matches = list(start.rglob(script_name))
    if not matches:
        raise FileNotFoundError(f'Não encontrei {script_name} a partir de {start}')
    return matches[0].parent

SOLUTION_DIR = find_solution_dir(SCRIPT_NAME)
DESAFIO_DIR = SOLUTION_DIR.parents[1]
INPUT_DIR = DESAFIO_DIR / 'data' / 'images'
OUTPUT_DIR = SOLUTION_DIR / 'dataset_yolo_sintetico'

sys.path.insert(0, str(SOLUTION_DIR))

from solucao_3_dados_sinteticos_detector_leve import brightness_variant, create_dataset, iter_images, parse_args, train_yolo, transform_boxes_for_flip, weak_boxes, yolo_line

print('Solução:', SOLUTION_DIR)
print('Imagens:', INPUT_DIR)
print('Saída  :', OUTPUT_DIR)

## 2. Escolha de imagem

Troque `image_index` para ver como os pseudo-rótulos se comportam em cada foto.

In [ ]:
image_paths = iter_images(INPUT_DIR)
print('Imagens encontradas:', len(image_paths))
for index, path in enumerate(image_paths):
    print(index, path.name)

image_index = 0
image_path = image_paths[image_index]
image_bgr = cv2.imread(str(image_path))
if image_bgr is None:
    raise ValueError(f'Não consegui ler {image_path}')

plt.figure(figsize=(8, 6))
plt.imshow(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
plt.title(image_path.name)
plt.axis('off')
plt.show()
print('Dimensão:', image_bgr.shape)

## 3. Parâmetros do pseudo-detector

Os valores abaixo controlam quais contornos viram caixas. Ajuste e execute novamente para comparar.

In [ ]:
min_area = 80.0
max_area = 20000.0
max_aspect_ratio = 8.0

args = parse_args([
    '--input-dir', str(INPUT_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--min-area', str(min_area),
    '--max-area', str(max_area),
    '--max-aspect-ratio', str(max_aspect_ratio),
])

boxes = weak_boxes(cv2, np, image_bgr, args)
print('Caixas encontradas:', len(boxes))
for box in boxes[:10]:
    print(box)

## 4. Visualização dos filtros clássicos

Esta célula replica a lógica de `weak_boxes` para mostrar onde os contornos surgem.

In [ ]:
gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
enhanced = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)
blurred = cv2.GaussianBlur(enhanced, (5, 5), 0)
_, mask = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
closed = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
opened = cv2.morphologyEx(closed, cv2.MORPH_OPEN, kernel, iterations=1)

stages = [
    ('Cinza', gray),
    ('CLAHE', enhanced),
    ('Blur', blurred),
    ('Otsu invertido', mask),
    ('Fechamento', closed),
    ('Abertura', opened),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for axis, (title, stage) in zip(axes.ravel(), stages):
    axis.imshow(stage, cmap='gray')
    axis.set_title(title)
    axis.axis('off')
plt.tight_layout()
plt.show()

## 5. Caixas sobre a imagem

Pseudo-rótulos são úteis para acelerar o trabalho, mas devem ser revisados. Nesta etapa avaliamos visualmente as caixas.

In [ ]:
annotated = image_bgr.copy()
for index, box in enumerate(boxes, start=1):
    cv2.rectangle(annotated, (box.x, box.y), (box.x + box.w, box.y + box.h), (0, 180, 0), 2)
    cv2.putText(annotated, str(index), (box.x, max(box.y - 5, 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1, cv2.LINE_AA)

plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.title(f'{len(boxes)} pseudo-caixas')
plt.axis('off')
plt.show()

## 6. Formato YOLO gerado

Cada linha do label YOLO usa `classe x_centro y_centro largura altura`, tudo normalizado entre 0 e 1.

In [ ]:
height, width = image_bgr.shape[:2]
label_lines = [yolo_line(box, width, height) for box in boxes]
print('\n'.join(label_lines[:10]) if label_lines else 'Nenhuma caixa gerada.')

## 7. Aumentações simples e correção das caixas

O script cria versões espelhadas e variações de brilho. Quando a imagem é espelhada, as caixas precisam ser transformadas também.

In [ ]:
height, width = image_bgr.shape[:2]
variants = [
    ('original', image_bgr, boxes),
    ('flip horizontal', cv2.flip(image_bgr, 1), transform_boxes_for_flip(boxes, width, height, 1)),
    ('flip vertical', cv2.flip(image_bgr, 0), transform_boxes_for_flip(boxes, width, height, 0)),
    ('mais escura', brightness_variant(cv2, np, image_bgr, 0.85, -10), boxes),
    ('mais clara', brightness_variant(cv2, np, image_bgr, 1.15, 12), boxes),
]

fig, axes = plt.subplots(1, len(variants), figsize=(4 * len(variants), 5))
for axis, (title, variant_image, variant_boxes) in zip(axes, variants):
    view = variant_image.copy()
    for box in variant_boxes:
        cv2.rectangle(view, (box.x, box.y), (box.x + box.w, box.y + box.h), (0, 180, 0), 2)
    axis.imshow(cv2.cvtColor(view, cv2.COLOR_BGR2RGB))
    axis.set_title(title)
    axis.axis('off')
plt.tight_layout()
plt.show()

## 8. Busca rápida de parâmetros

Sem rótulo real, a melhor checagem é visual. Ainda assim, contar quantas caixas cada configuração gera ajuda a identificar valores instáveis.

In [ ]:
from itertools import product

search_space = {
    'min_area': [50, 80, 120, 200],
    'max_area': [8000, 12000, 20000],
    'max_aspect_ratio': [4.0, 6.0, 8.0],
}

sample_images = image_paths[:5]
keys = list(search_space.keys())
rows = []

for values in product(*(search_space[key] for key in keys)):
    params = dict(zip(keys, values))
    test_args = parse_args([
        '--input-dir', str(INPUT_DIR),
        '--output-dir', str(OUTPUT_DIR),
        '--min-area', str(params['min_area']),
        '--max-area', str(params['max_area']),
        '--max-aspect-ratio', str(params['max_aspect_ratio']),
    ])
    counts = []
    for path in sample_images:
        img = cv2.imread(str(path))
        counts.append(len(weak_boxes(cv2, np, img, test_args)))
    rows.append({**params, 'counts': counts, 'mean_count': float(np.mean(counts)), 'std_count': float(np.std(counts))})

rows = sorted(rows, key=lambda row: (row['std_count'], -row['mean_count']))
for row in rows[:10]:
    print(row)

## 9. Criar o dataset YOLO sintético

Por segurança, a geração fica desligada. Mude `execute_dataset_creation` para `True` para criar `dataset_yolo_sintetico`.

In [ ]:
execute_dataset_creation = False

args_dataset = parse_args([
    '--input-dir', str(INPUT_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--seed', '42',
    '--val-fraction', '0.2',
    '--min-area', str(min_area),
    '--max-area', str(max_area),
    '--max-aspect-ratio', str(max_aspect_ratio),
])

if execute_dataset_creation:
    data_yaml = create_dataset(args_dataset)
    print('Arquivo YOLO:', data_yaml)
else:
    print('Geração desativada. Mude execute_dataset_creation = True para criar o dataset.')

## 10. Inspecionar o dataset gerado

Depois de criar o dataset, visualize uma imagem e leia o label correspondente.

In [ ]:
generated_images = sorted((OUTPUT_DIR / 'images' / 'train').glob('*.jpg'))

if not generated_images:
    print('Dataset ainda não foi gerado.')
else:
    generated_image_path = generated_images[0]
    generated_label_path = OUTPUT_DIR / 'labels' / 'train' / f'{generated_image_path.stem}.txt'
    generated_image = cv2.imread(str(generated_image_path))
    h, w = generated_image.shape[:2]
    view = generated_image.copy()

    for line in generated_label_path.read_text(encoding='utf-8').splitlines():
        cls, xc, yc, bw, bh = line.split()
        xc, yc, bw, bh = map(float, [xc, yc, bw, bh])
        x1 = int((xc - bw / 2) * w)
        y1 = int((yc - bh / 2) * h)
        x2 = int((xc + bw / 2) * w)
        y2 = int((yc + bh / 2) * h)
        cv2.rectangle(view, (x1, y1), (x2, y2), (0, 180, 0), 2)

    plt.figure(figsize=(8, 6))
    plt.imshow(cv2.cvtColor(view, cv2.COLOR_BGR2RGB))
    plt.title(generated_image_path.name)
    plt.axis('off')
    plt.show()
    print(generated_label_path.read_text(encoding='utf-8')[:1000])

## 11. Treino YOLO opcional

O treino pode baixar pesos e demorar. Execute apenas depois de revisar os pseudo-rótulos.

In [ ]:
execute_yolo_training = False
data_yaml = OUTPUT_DIR / 'data.yaml'

args_train = parse_args([
    '--input-dir', str(INPUT_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--model', 'yolov8n.pt',
    '--epochs', '5',
    '--imgsz', '640',
    '--batch', '4',
])

if execute_yolo_training:
    if not data_yaml.exists():
        data_yaml = create_dataset(args_dataset)
    train_yolo(data_yaml, args_train)
else:
    print('Treino YOLO desativado. Revise os labels antes de mudar para True.')

## 12. Dicas de estudo

- Se muitos ruídos viram caixas, aumente `min_area` ou reduza `max_aspect_ratio`.
- Se parafusos grandes somem, aumente `max_area`.
- Espelhamentos precisam transformar caixas; brilho não muda coordenadas.
- Pseudo-rótulo não substitui revisão humana. Use o dataset gerado como ponto de partida.
- Antes de treinar YOLO, abra algumas imagens de treino e validação para confirmar a qualidade das caixas.